# 03 — LangChain Hello RAG

**Goal**: Build the smallest possible RAG in Python — to feel what LangChain does vs Semantic Kernel.

Steps: load text → chunk → embed → store in FAISS → ask a question.

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

## 1. Sample documents

In [ ]:
docs = [
    "Semantic Kernel is Microsoft's open-source SDK for building LLM-powered apps in C#, Python and Java.",
    "Azure AI Search supports vector search using HNSW and DiskANN algorithms for fast nearest-neighbor lookup.",
    "RAG (Retrieval Augmented Generation) reduces hallucinations by grounding LLM answers in retrieved documents.",
    "text-embedding-3-small produces 1536-dimensional vectors and is cheaper than text-embedding-3-large.",
    "Azure Functions is a serverless compute service ideal for event-driven workloads like chatbots.",
]

## 2. Embed + store in FAISS (in-memory vector DB)

In [ ]:
from langchain_openai import AzureOpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    azure_deployment=os.getenv("AZURE_OPENAI_EMBED_DEPLOYMENT"),
    api_version="2024-02-15-preview",
)

store = FAISS.from_texts(docs, embeddings)
print("Indexed", len(docs), "documents")

## 3. Retrieve + answer

In [ ]:
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    azure_deployment=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT"),
    api_version="2024-02-15-preview",
    temperature=0,
)

prompt = ChatPromptTemplate.from_template(
    "Answer using ONLY the context. If unsure say 'I don't know'.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)

def ask(question: str):
    hits = store.similarity_search(question, k=3)
    context = "\n".join(f"- {h.page_content}" for h in hits)
    chain = prompt | llm
    return chain.invoke({"context": context, "question": question}).content

print(ask("What dimension are text-embedding-3-small vectors?"))
print("---")
print(ask("What is RAG?"))
print("---")
print(ask("What's the capital of France?"))   # should say I don't know

## ✅ Done

That's a complete RAG in ~30 lines. Compare it mentally to what you'll build in Weeks 7-8 with .NET — same flow, different syntax.

**Next**: [04-huggingface-embeddings.ipynb](04-huggingface-embeddings.ipynb)